In [8]:
# import numpy as np
# from sklearn.neighbors import NearestNeighbors
# from collections import defaultdict
# import heapq

# class HubsGraph:
#     def __init__(self, vectors, hub_count=50, k_neighbors=10, h_hubs=3, hub_connections=5):
#         """
#         Initialize the Hubs Graph
        
#         :param vectors: Dataset vectors (numpy array)
#         :param hub_count: Number of hubs to identify
#         :param k_neighbors: Number of nearest neighbor connections per node
#         :param h_hubs: Number of hub connections per node
#         :param hub_connections: Number of hub-to-hub connections per hub
#         """
#         self.vectors = vectors
#         self.hub_count = hub_count
#         self.k_neighbors = k_neighbors
#         self.h_hubs = h_hubs
#         self.hub_connections = hub_connections
#         self.hub_scores = None
#         self.hubs = None
#         self.graph = defaultdict(list)
        
#     def fit(self, training_queries):
#         """
#         Learn hub nodes from training queries
        
#         :param training_queries: List of (query_vector, nearest_neighbors) pairs
#         """
#         # Calculate hub scores based on query result frequency
#         hub_scores = defaultdict(int)
#         for _, neighbors in training_queries:
#             for neighbor in neighbors:
#                 hub_scores[neighbor] += 1
                
#         # Convert to numpy array format
#         scores = np.zeros(len(self.vectors))
#         for idx, count in hub_scores.items():
#             scores[idx] = count
            
#         self.hub_scores = scores
#         self.hubs = np.argsort(scores)[-self.hub_count:][::-1]
        
#     def build_graph(self):
#         """Construct the graph with hub connections"""
#         # Find k-nearest neighbors for all nodes
#         nbrs = NearestNeighbors(n_neighbors=self.k_neighbors+1).fit(self.vectors)
#         _, knn_indices = nbrs.kneighbors(self.vectors)
        
#         # Find nearest hubs for all nodes
#         hub_vecs = self.vectors[self.hubs]
#         hub_nbrs = NearestNeighbors(n_neighbors=self.h_hubs).fit(hub_vecs)
#         _, hub_indices = hub_nbrs.kneighbors(self.vectors)
#         hub_indices = self.hubs[hub_indices]
        
#         # Build base graph
#         for i in range(len(self.vectors)):
#             # Get KNN connections (excluding self)
#             neighbors = [idx for idx in knn_indices[i] if idx != i]
            
#             # Get hub connections
#             # hubs = list(hub_indices[i])
#             hubs = hub_indices[i].tolist()
            
#             # Combine connections
#             all_connections = list(set(neighbors + hubs))
#             self.graph[i] = all_connections
            
#         # Add hub-to-hub connections
#         hub_vecs = self.vectors[self.hubs]
#         hub_nbrs = NearestNeighbors(n_neighbors=self.hub_connections+1).fit(hub_vecs)
#         _, hub_hub_indices = hub_nbrs.kneighbors(hub_vecs)
        
#         for i, hub in enumerate(self.hubs):
#             # Get nearest hubs (excluding self)
#             connections = [self.hubs[idx] for idx in hub_hub_indices[i][1:]]
            
#             # Add bidirectional connections
#             self.graph[hub] = list(set(self.graph[hub] + connections))
#             for conn in connections:
#                 self.graph[conn].append(hub)
                
#         # Remove duplicates
#         for k in self.graph:
#             self.graph[k] = list(set(self.graph[k]))
            
#     def search(self, query, ef=100, k=10):
#         """
#         Search the graph for nearest neighbors
        
#         :param query: Query vector
#         :param ef: Exploration factor
#         :param k: Number of results to return
#         :return: Indices of nearest neighbors
#         """
#         # Start from random hub
#         entry_point = np.random.choice(self.hubs)
        
#         # Priority queue sorted by distance
#         candidates = [(np.linalg.norm(query - self.vectors[entry_point]), entry_point)]
#         visited = set()
#         results = []
        
#         while candidates:
#             dist, node = heapq.heappop(candidates)
            
#             if node in visited:
#                 continue
                
#             visited.add(node)
            
#             # Add to results
#             heapq.heappush(results, (-dist, node))
#             if len(results) > ef:
#                 heapq.heappop(results)
                
#             # Explore neighbors
#             for neighbor in self.graph[node]:
#                 if neighbor not in visited:
#                     new_dist = np.linalg.norm(query - self.vectors[neighbor])
#                     heapq.heappush(candidates, (new_dist, neighbor))
                    
#         # Return top k results
#         return [node for _, node in sorted(results, reverse=True)[:k]]
    
#     def get_hubs(self):
#         """Return list of hub indices"""
#         return self.hubs
    
#     def get_graph(self):
#         """Return adjacency list representation of graph"""
#         return self.graph

# # Example usage
# if __name__ == "__main__":
#     # Generate sample data
#     np.random.seed(42)
#     data = np.random.randn(1000, 128)  # 1000 128-dim vectors
    
#     # Generate sample training queries (mock)
#     training_queries = []
#     for _ in range(100):
#         query = np.random.randn(128)
#         # Mock nearest neighbors (top 10 random points)
#         neighbors = np.random.choice(1000, 10, replace=False).tolist()
#         training_queries.append( (query, neighbors) )
    
#     # Initialize and build graph
#     graph = HubsGraph(data, hub_count=50, k_neighbors=10, h_hubs=3)
#     graph.fit(training_queries)
#     graph.build_graph()
    
#     # Perform a sample query
#     query = np.random.randn(128)
#     results = graph.search(query)
#     print("Top results:", results)
#     print("Identified hubs:", graph.get_hubs()[:10])

# SIFT adapt

In [9]:
import numpy as np
import struct
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict
import heapq
import time
from tqdm.auto import tqdm

class HubsGraph:
    def __init__(self, base_vectors, hub_count=100, k_neighbors=15, h_hubs=5, hub_connections=10):
        self.vectors = base_vectors
        self.hub_count = hub_count
        self.k_neighbors = k_neighbors
        self.h_hubs = h_hubs
        self.hub_connections = hub_connections
        self.hub_scores = None
        self.hubs = None
        self.graph = defaultdict(list)
        
    def fit(self, learn_vectors, num_queries=10000):
        # Find actual neighbors for training queries
        nbrs = NearestNeighbors(n_neighbors=100).fit(self.vectors)
        _, neighbors = nbrs.kneighbors(learn_vectors[:num_queries])
        
        # Calculate hub scores with exponential decay
        hub_scores = defaultdict(float)
        for query_neighbors in tqdm(neighbors, desc="Processing training queries"):
            for rank, idx in enumerate(query_neighbors):
                hub_scores[idx] += np.exp(-rank/5)
                
        scores = np.zeros(len(self.vectors))
        for idx, count in hub_scores.items():
            scores[idx] = count
            
        self.hub_scores = scores
        self.hubs = np.argsort(scores)[-self.hub_count:][::-1]
        
    def build_graph(self):
        # Find k-nearest neighbors
        nbrs = NearestNeighbors(n_neighbors=self.k_neighbors+1).fit(self.vectors)
        _, knn_indices = nbrs.kneighbors(self.vectors)
        
        # Find nearest hubs for all nodes
        hub_vecs = self.vectors[self.hubs]
        hub_nbrs = NearestNeighbors(n_neighbors=self.h_hubs).fit(hub_vecs)
        _, hub_indices = hub_nbrs.kneighbors(self.vectors)
        hub_indices = self.hubs[hub_indices]
        
        # Build base graph
        for i in tqdm(range(len(self.vectors)), desc="Connecting nodes"):
            # KNN connections (excluding self)
            neighbors = [int(idx) for idx in knn_indices[i] if idx != i]
            
            # Hub connections
            hubs = hub_indices[i].tolist()
            
            # Combine connections
            all_connections = list(set(neighbors + hubs))
            self.graph[i] = all_connections
            
        # Add hub-to-hub connections
        hub_nbrs = NearestNeighbors(n_neighbors=self.hub_connections+1).fit(hub_vecs)
        _, hub_hub_indices = hub_nbrs.kneighbors(hub_vecs)
        
        for i, hub in tqdm(enumerate(self.hubs), desc="Connecting hubs"):
            connections = [int(self.hubs[idx]) for idx in hub_hub_indices[i][1:]]
            self.graph[hub] = list(set(self.graph[hub] + connections))
            for conn in connections:
                if hub not in self.graph[conn]:
                    self.graph[conn].append(hub)
                    
        # Add reverse connections
        for i in tqdm(list(self.graph.keys()), desc="Adding reverse links"):
            for neighbor in self.graph[i]:
                if i not in self.graph[neighbor]:
                    self.graph[neighbor].append(i)
        
    def search(self, query, ef=200, k=100):
        # Find closest hub as entry point
        hub_vecs = self.vectors[self.hubs]
        distances = np.linalg.norm(hub_vecs - query, axis=1)
        entry_point = self.hubs[np.argmin(distances)]
        
        candidates = [(np.linalg.norm(query - self.vectors[entry_point]), entry_point)]
        visited = set()
        results = []
        
        while candidates:
            dist, node = heapq.heappop(candidates)
            
            if node in visited:
                continue
                
            visited.add(node)
            
            heapq.heappush(results, (-dist, node))
            if len(results) > ef:
                heapq.heappop(results)
                
            for neighbor in self.graph[node]:
                if neighbor not in visited:
                    new_dist = np.linalg.norm(query - self.vectors[neighbor])
                    heapq.heappush(candidates, (new_dist, neighbor))
                    
        return [node for _, node in sorted(results, reverse=True)[:k]]
    
    def evaluate(self, queries, ground_truth, k=100):
        recalls = []
        latencies = []
        
        for query, true_neighbors in tqdm(zip(queries, ground_truth), 
                                        total=len(queries),
                                        desc="Evaluating queries"):
            start_time = time.time()
            predicted = self.search(query, k=k)
            latencies.append(time.time() - start_time)
            
            recall = len(set(predicted) & set(true_neighbors)) / len(true_neighbors)
            recalls.append(recall)
            
        return {
            'mean_recall': np.mean(recalls),
            'p95_recall': np.percentile(recalls, 95),
            'mean_latency': np.mean(latencies),
            'p95_latency': np.percentile(latencies, 95)
        }

# Data loading functions
def read_fvecs(fp):
    with open(fp, "rb") as f:
        vecs = []
        while True:
            header = f.read(4)
            if len(header) == 0: break
            dim = struct.unpack('i', header)[0]
            vec = struct.unpack('f'*dim, f.read(4*dim))
            vecs.append(np.array(vec))
        return np.vstack(vecs)

def read_ivecs(fp):
    with open(fp, "rb") as f:
        vecs = []
        while True:
            header = f.read(4)
            if len(header) == 0: break
            dim = struct.unpack('i', header)[0]
            vec = struct.unpack('i'*dim, f.read(4*dim))
            vecs.append(np.array(vec))
        return np.vstack(vecs)

# Example usage
if __name__ == "__main__":
    # Load SIFT dataset
    base = read_fvecs('/Users/na/CSE Sem Store/Data Science/Project/siftsmall/siftsmall_base.fvecs')
    learn = read_fvecs('/Users/na/CSE Sem Store/Data Science/Project/siftsmall/siftsmall_learn.fvecs')
    queries = read_fvecs('/Users/na/CSE Sem Store/Data Science/Project/siftsmall/siftsmall_query.fvecs')
    ground_truth = read_ivecs('/Users/na/CSE Sem Store/Data Science/Project/siftsmall/siftsmall_groundtruth.ivecs')

    # Normalize vectors
    base = base / np.linalg.norm(base, axis=1)[:, np.newaxis]
    learn = learn / np.linalg.norm(learn, axis=1)[:, np.newaxis]
    queries = queries / np.linalg.norm(queries, axis=1)[:, np.newaxis]

    # Create and train graph
    graph = HubsGraph(base, hub_count=100, k_neighbors=15, h_hubs=5)
    graph.fit(learn)
    graph.build_graph()

    # Evaluate
    metrics = graph.evaluate(queries, ground_truth[:, :100])
    print(f"Mean Recall@100: {metrics['mean_recall']:.3f}")
    print(f"P95 Latency: {metrics['p95_latency']:.4f}s")

Connecting nodes: 100%|██████████| 10000/10000 [00:00<00:00, 30510.69it/s]
Connecting hubs: 100it [00:00, 131730.65it/s]
Evaluating queries: 100%|██████████| 100/100 [00:32<00:00,  3.07it/s]

Mean Recall@100: 0.995
P95 Latency: 0.3640s
